Read CSV

In [0]:
SELECT COUNT(*) AS total_rows
FROM read_files(
  '/Volumes/nestle_dev_bronze/data_lake/landing/sql_sales_transactions_day3.csv',
  format => 'csv',
  header => true
)

Get Watermark

In [0]:
SELECT last_high_water_mark AS last_watermark
FROM nestle_dev_silver.control.watermark_tracking
WHERE source_id = 'sql_sales_transactions'

 Filter & Load (Only modified + new)

In [0]:
INSERT INTO nestle_dev_bronze.sql_server.sales_transactions
  (transaction_id, product_id, region, channel, customer_id, quantity, unit_price, amount, created_at, modified_at, ingestion_timestamp)
SELECT
  CAST(transaction_id AS STRING)  AS transaction_id,
  product_id,
  region,
  channel,
  customer_id,
  CAST(quantity AS BIGINT)        AS quantity,
  unit_price,
  amount,
  CAST(created_at AS STRING)      AS created_at,
  CAST(modified_at AS STRING)     AS modified_at,
  current_timestamp()             AS ingestion_timestamp
FROM read_files(
  '/Volumes/nestle_dev_bronze/data_lake/landing/sql_sales_transactions_day3.csv',
  format => 'csv',
  header => true
)
WHERE modified_at > (
  SELECT last_high_water_mark
  FROM nestle_dev_silver.control.watermark_tracking
  WHERE source_id = 'sql_sales_transactions'
)

Verify Total

In [0]:
SELECT
  COUNT(*) AS total_rows,
  '400 (370 Day1+Day2 + 30 Day3)' AS expected
FROM nestle_dev_bronze.sql_server.sales_transactions